<a href="https://colab.research.google.com/github/JaniceKC/salesinsight-py/blob/main/salesinsight_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from datetime import datetime, timedelta
import random

# =========================
# RF01 – Gerar Dataset
# =========================
def gerar_dataset_vendas(n_registros=200, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor", "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares", "Tablet": "Celulares",
                  "Monitor": "Computadores", "Teclado": "Periféricos", "Mouse": "Periféricos",
                  "Headset": "Periféricos"}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    clientes = [f"Cliente_{i:03d}" for i in range(1, 51)]

    data_inicio = datetime(2024, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco_base = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
                      "Monitor": 1200, "Teclado": 250, "Mouse": 120, "Headset": 350}[produto]
        preco = round(preco_base * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))

        if random.random() < 0.05: quantidade = None
        if random.random() < 0.04: preco = None
        if random.random() < 0.03: produto = "  " + produto

        dados.append({
            "id_venda": i + 1,
            "data_venda": data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVÁLIDA",
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })

    return pd.DataFrame(dados)

# =========================
# RF03 – Limpeza
# =========================
def limpar_dados(df):
    df = df.copy()
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    df = df.dropna(subset=["data_venda", "quantidade", "preco_unitario"])
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)
    return df

# =========================
# RF04 – Colunas Derivadas
# =========================
def criar_colunas_derivadas(df):
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]
    df["mes"] = df["data_venda"].dt.month
    df["trimestre"] = df["data_venda"].dt.quarter
    df["ano"] = df["data_venda"].dt.year
    return df

# =========================
# RF05 – Métricas
# =========================
def calcular_metricas(df):
    metricas = {}
    metricas["por_mes"] = df.groupby("mes")["receita_total"].sum().reset_index()
    metricas["top_produtos"] = df.groupby("produto")["receita_total"].sum().nlargest(5).reset_index()
    metricas["por_categoria"] = df.groupby("categoria")["receita_total"].sum().reset_index()
    metricas["por_regiao"] = df.groupby("regiao")["receita_total"].sum().reset_index()
    return metricas

# =========================
# RF06 – Segmentação
# =========================
def segmentar_clientes(df):
    clientes = df.groupby("cliente")["receita_total"].sum().reset_index()
    clientes["segmento"] = clientes["receita_total"].apply(
        lambda x: "Ouro" if x > 15000 else ("Prata" if x >= 5000 else "Bronze")
    )
    return clientes

# =========================
# RF08 – Visualizações
# =========================
def gerar_visualizacoes(df, metricas):
    os.makedirs("outputs/graficos", exist_ok=True)
    sns.lineplot(data=metricas["por_mes"], x="mes", y="receita_total")
    plt.title("Receita por Mês")
    plt.savefig("outputs/graficos/vendas_por_mes.png")
    plt.close()

    sns.barplot(data=metricas["top_produtos"], x="receita_total", y="produto")
    plt.title("Top 5 Produtos")
    plt.savefig("outputs/graficos/top_produtos.png")
    plt.close()

    sns.boxplot(data=df, x="regiao", y="receita_total")
    plt.title("Distribuição por Região")
    plt.savefig("outputs/graficos/distribuicao_regioes.png")
    plt.close()

# =========================
# RF09 – Classe Pipeline
# =========================
class AnalisadorDeVendas:
    def __init__(self, caminho_csv):
        self.df = pd.read_csv(caminho_csv)
        self.metricas = None
        self.clientes = None

    def executar(self):
        self.df = limpar_dados(self.df)
        self.df = criar_colunas_derivadas(self.df)
        self.metricas = calcular_metricas(self.df)
        self.clientes = segmentar_clientes(self.df)
        gerar_visualizacoes(self.df, self.metricas)
        print("Pipeline concluído com sucesso!")

# =========================
# RF14 – Ponto de Entrada
# =========================
if __name__ == "__main__":
    df = gerar_dataset_vendas()
    df.to_csv("vendas.csv", index=False)
    analisador = AnalisadorDeVendas("vendas.csv")
    analisador.executar()

/tmp/ipykernel_9194/849249744.py:57: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


Pipeline concluído com sucesso!
